# 🚀 Image Enhancer — Colab GPU Worker

**Setup (do this once):**
1. `Runtime → Change runtime type → T4 GPU` → Save
2. `Runtime → Run all` (`Ctrl+F9`)
3. Wait for Cell 1 to finish installing (~2 min)
4. Cell 4 will print a public URL like `https://xxxx.gradio.live`
5. Copy that URL and paste it into your local app's **Remote URL** field

> ⚠️ **Keep this tab open and active.** Colab disconnects idle sessions after ~90 min.  
> Cell 5 runs a keep-alive loop that prevents idle disconnection.

In [ ]:
# ── Cell 1: Verify GPU & Install Dependencies ─────────────────────────────
import subprocess, sys

# Confirm GPU is available before wasting time installing
gpu_check = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                           capture_output=True, text=True)
if gpu_check.returncode == 0:
    print(f'✅ GPU detected: {gpu_check.stdout.strip()}')
else:
    print('❌ NO GPU DETECTED!')
    print('   Go to Runtime → Change runtime type → T4 GPU → Save, then re-run.')
    raise SystemExit('Stopping — GPU required.')

print('\nInstalling dependencies (this takes ~2 minutes)...')

# Install in correct order — torch must come before basicsr/realesrgan
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
    'gradio==4.44.1', 'gradio-client', 'numpy<2', 'pillow', 'opencv-python-headless', 'psutil'])

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
    'torch==2.2.0', 'torchvision==0.17.0',
    '--index-url', 'https://download.pytorch.org/whl/cu118'])

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
    'basicsr', 'realesrgan'])

print('\n✅ All dependencies installed.')

In [ ]:
# ── Cell 2: Compatibility Shims (Python 3.12 / modern torchvision) ─────────
import sys, types

# basicsr still imports distutils.version which was removed in Python 3.12
if 'distutils.version' not in sys.modules:
    try:
        from setuptools._distutils.version import LooseVersion
    except ImportError:
        from packaging.version import parse as LooseVersion
    distutils_mod = types.ModuleType('distutils')
    version_mod   = types.ModuleType('distutils.version')
    version_mod.LooseVersion = LooseVersion
    distutils_mod.version    = version_mod
    sys.modules['distutils']         = distutils_mod
    sys.modules['distutils.version'] = version_mod

# torchvision removed functional_tensor in newer versions
if 'torchvision.transforms.functional_tensor' not in sys.modules:
    try:
        import torchvision.transforms.functional as _tf
        sys.modules['torchvision.transforms.functional_tensor'] = _tf
    except ImportError:
        pass

print('✅ Compatibility shims applied.')

In [ ]:
# ── Cell 3: Load Model onto GPU ────────────────────────────────────────────
import os, time
import numpy as np
import torch
import cv2
from PIL import Image
from urllib.request import urlretrieve
from basicsr.archs.rrdbnet_arch import RRDBNet
from realesrgan import RealESRGANer

os.makedirs('models', exist_ok=True)

# ── Confirm CUDA ──────────────────────────────────────────────────────────
assert torch.cuda.is_available(), 'CUDA not available — did you set GPU runtime?'
device = torch.device('cuda:0')
print(f'✅ Using device: {torch.cuda.get_device_name(0)}')
print(f'   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')

# ── Download model weights if not cached ─────────────────────────────────
MODEL_PATH = 'models/RealESRGAN_x4plus.pth'
if not os.path.exists(MODEL_PATH):
    print('Downloading RealESRGAN_x4plus.pth (~64 MB)...')
    urlretrieve(
        'https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth',
        MODEL_PATH
    )
    print('✅ Model downloaded.')
else:
    print('✅ Model weights already cached.')

# ── Load model — cached as module-level singleton ─────────────────────────
# tile=0 means full image in one pass (GPU has enough VRAM).
# If you get OOM errors, change tile=256 or tile=128.
_net = RRDBNet(num_in_ch=3, num_out_ch=3, num_feat=64,
               num_block=23, num_grow_ch=32, scale=4)

_upsampler = RealESRGANer(
    scale=4,
    model_path=MODEL_PATH,
    model=_net,
    tile=0,          # 0 = full image on GPU (fastest). Change to 256 if OOM.
    tile_pad=10,
    pre_pad=0,
    half=True,       # float16 on GPU — 2× faster, half the VRAM
    device=device,
)

print(f'✅ RealESRGAN loaded on GPU (half precision = {_upsampler.half})')
print(f'   Model parameters: {sum(p.numel() for p in _net.parameters()):,}')

In [ ]:
# ── Cell 4: Define API Function & Launch Gradio Server ────────────────────
import gradio as gr
import psutil, json

def enhance(
    image: np.ndarray,
    method: str  = 'realesrgan',
    scale: int   = 4,
    tile: int    = 0,
    face_enhance: bool = False,
    progress = gr.Progress(track_tqdm=True),
) -> np.ndarray:
    """
    GPU-accelerated image enhancement.
    Called by gradio_client from the local Flask app.

    Args:
        image        : RGB uint8 numpy array (Gradio decodes the uploaded file)
        method       : 'realesrgan', 'bicubic', or 'lanczos'
        scale        : upscale factor — 2, 4, or 8
        tile         : tile size for VRAM control (0 = full image)
        face_enhance : reserved — not implemented in this worker

    Returns:
        Enhanced RGB uint8 numpy array.
    """
    scale  = int(scale)
    tile   = int(tile)
    method = str(method).strip().lower()

    h, w = image.shape[:2]
    print(f'\n[Worker] Received: method={method}  scale={scale}x  tile={tile}  size={w}×{h}')
    t0 = time.time()

    # ── GPU memory info before processing ────────────────────────────────
    vram_free  = torch.cuda.mem_get_info()[0] / 1024**3
    vram_total = torch.cuda.mem_get_info()[1] / 1024**3
    print(f'[Worker] VRAM: {vram_free:.1f}/{vram_total:.1f} GB free')

    if method == 'realesrgan':
        # ── Reconfigure tile size if caller requested a specific one ──────
        if tile != _upsampler.tile_size:
            _upsampler.tile_size = tile
            print(f'[Worker] Tile size updated to {tile}')

        # ── Calculate total tiles for progress reporting ──────────────────
        if tile > 0:
            y_steps = len(range(0, h, tile))
            x_steps = len(range(0, w, tile))
            total_tiles = y_steps * x_steps
        else:
            total_tiles = 1

        tile_count = {'n': 0}
        original_model = _upsampler.model

        # Wrap the model to intercept each forward pass and emit progress
        class _ProgressWrapper:
            def __call__(self, *args, **kwargs):
                result = original_model(*args, **kwargs)
                tile_count['n'] += 1
                frac = tile_count['n'] / total_tiles
                try:
                    gpu_util = torch.cuda.utilization()
                except Exception:
                    gpu_util = 0
                progress(frac, desc=json.dumps({
                    'current': tile_count['n'],
                    'total':   total_tiles,
                    'dt':      round(time.time() - t0, 2),
                    'cpu':     gpu_util,   # repurpose cpu field for GPU util %
                }))
                return result
            def __getattr__(self, name):
                return getattr(original_model, name)

        try:
            _upsampler.model = _ProgressWrapper()
            bgr_in  = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)
            bgr_out, _ = _upsampler.enhance(bgr_in, outscale=scale)
            result  = cv2.cvtColor(bgr_out, cv2.COLOR_BGR2RGB)
        finally:
            _upsampler.model = original_model   # always restore
            torch.cuda.empty_cache()            # free VRAM after each request

    elif method in ('bicubic', 'lanczos'):
        # Classical interpolation — runs on CPU, still useful for comparison
        pil_img  = Image.fromarray(image)
        resample = Image.BICUBIC if method == 'bicubic' else Image.LANCZOS
        result   = np.array(pil_img.resize((w * scale, h * scale), resample))

    else:
        raise ValueError(f'Unknown method: {method!r}. Use realesrgan, bicubic, or lanczos.')

    elapsed = time.time() - t0
    out_h, out_w = result.shape[:2]
    print(f'[Worker] ✅ Done in {elapsed:.2f}s  output={out_w}×{out_h}')
    return result


# ── Build Gradio interface ────────────────────────────────────────────────
with gr.Blocks(title='Image Enhancer — Colab GPU Worker') as demo:
    gr.Markdown(
        '## 🚀 Image Enhancer — Colab GPU Worker\n'
        'This server is called automatically by your local Flask app.  \n'
        'You can also test it manually below.'
    )
    with gr.Row():
        with gr.Column():
            img_in       = gr.Image(type='numpy', label='Input Image')
            method_in    = gr.Radio(['realesrgan', 'bicubic', 'lanczos'],
                                    value='realesrgan', label='Method')
            scale_in     = gr.Slider(minimum=2, maximum=8, step=2,
                                     value=4, label='Scale Factor')
            tile_in      = gr.Slider(minimum=0, maximum=512, step=64,
                                     value=0, label='Tile Size (0 = full GPU)')
            face_in      = gr.Checkbox(value=False, label='Face Enhance (reserved)')
            run_btn      = gr.Button('▶ Enhance', variant='primary')
        with gr.Column():
            img_out = gr.Image(type='numpy', label='Enhanced Output')

    run_btn.click(
        fn=enhance,
        inputs=[img_in, method_in, scale_in, tile_in, face_in],
        outputs=img_out,
        api_name='enhance',   # ← gradio_client calls /enhance
    )

# queue() is required for API calls and concurrent requests
demo.queue(max_size=4)

print('\n' + '='*60)
print('Starting Gradio server...')
print('='*60)

demo.launch(
    share=True,       # generates public https://xxxx.gradio.live URL
    debug=False,
    show_error=True,
    quiet=False,
    inline=False,     # don't embed iframe in Colab output
    prevent_thread_lock=True,   # don't block — keep-alive loop runs in Cell 5
)

print('\n✅ Server launched.')
print('👆 Copy the gradio.live URL above and paste it into your local app.')

In [ ]:
# ── Cell 5: Keep-Alive Loop ────────────────────────────────────────────────
# Colab disconnects sessions that appear idle.
# This loop prints a heartbeat every 30 seconds and does a tiny GPU op
# to keep the runtime active. It runs forever — stop it manually if needed.

import time, datetime, torch

print('🔄 Keep-alive loop started. This cell runs forever to prevent disconnection.')
print('   Stop it manually (■ button) only when you are done.')
print()

heartbeat_interval = 30   # seconds between heartbeats
tick = 0

while True:
    tick += 1
    now = datetime.datetime.now().strftime('%H:%M:%S')

    # Tiny GPU op — keeps CUDA context alive and prevents idle timeout
    _ = torch.zeros(1, device='cuda').sum()

    # System stats
    vram_free  = torch.cuda.mem_get_info()[0] / 1024**3
    vram_total = torch.cuda.mem_get_info()[1] / 1024**3
    vram_used  = vram_total - vram_free
    ram_pct    = __import__('psutil').virtual_memory().percent

    print(f'[{now}] ❤️  tick={tick:04d}  '
          f'VRAM {vram_used:.1f}/{vram_total:.1f}GB  '
          f'RAM {ram_pct:.0f}%  '
          f'Server: running ✅',
          flush=True)

    time.sleep(heartbeat_interval)